In [2]:
# 1. Libraries

import pandas as pd
import dash
from dash import dcc, html, Input, Output, State
import dash_ag_grid as dag
import dash_bootstrap_components as dbc
import plotly.express as px
from plotly.subplots import make_subplots


In [3]:
# 2. Import File
path_to_file = "https://raw.githubusercontent.com/cesarlarasantana/VDS_2526_G04_Football/refs/heads/main/Datasets"

df_country = pd.read_csv(path_to_file+"/Country.csv")
df_match_goals = pd.read_csv(path_to_file+"/Match_Goals.csv")
df_match_shots_on = pd.read_csv(path_to_file+"/Match_Shots_On.csv")
df_match_fouls = pd.read_csv(path_to_file+"/Match_Fouls_Committed.csv")
df_match_cards = pd.read_csv(path_to_file+"/Match_Cards.csv")
df_team = pd.read_csv(path_to_file+"/Team.csv")
df_match = pd.read_csv(path_to_file+"/Match.csv")
df_player = pd.read_csv(path_to_file+"/Player.csv")
df_player_att = pd.read_csv(path_to_file+"/Player_Attributes.csv", sep= ';')
df_position_ref = pd.read_csv(path_to_file+"/PositionReference.csv")


C:\Users\pci\AppData\Local\Temp\ipykernel_2736\4289516213.py:7: DtypeWarning: Columns (0: player1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_match_fouls = pd.read_csv(path_to_file+"/Match_Fouls_Committed.csv")


In [4]:
## Data Preparation

df_player_example = df_player_att[df_player_att['player_api_id'] == 23499]
agg_df_player_example = df_player_example.groupby('player_api_id').agg(
    {'finishing' : 'mean',
     'crossing'  : 'mean',
     'ball_control'  : 'mean',
     'agility'  : 'mean',
     'stamina'  : 'mean',
     'strength'  : 'mean'}
).reset_index()

df_player_benchmark = df_player_att[df_player_att['player_api_id'] != 23499]
agg_df_player_benchmark = df_player_benchmark.groupby('player_api_id').agg(
    {'finishing' : 'mean',
     'crossing'  : 'mean',
     'ball_control'  : 'mean',
     'agility'  : 'mean',
     'stamina'  : 'mean',
     'strength'  : 'mean'}
).reset_index()

colors_bar = ["#1D9B28", "#49C718", "#9DEC1C", "#2439F3", "#0E56F1A9", "#5AD5EB" ]


In [5]:
# Calculate angles for each slice
vars_play = ["finishing", "crossing", "ball_control", "agility", "stamina", "strength"]

values_to_chart = []
vec_cols_agg_df_player = agg_df_player_example.columns[1:].to_list()
values_benchmark = []

for i in range(0, len(vars_play)):
    for j in range(0, len(vec_cols_agg_df_player)):
        if vars_play[i] == vec_cols_agg_df_player[j]:
            values_to_chart.append(agg_df_player_example.iloc[0][vec_cols_agg_df_player[j]])

for i in range(0, len(vars_play)):
    for j in range(0, len(vec_cols_agg_df_player)):
        if vars_play[i] == vec_cols_agg_df_player[j]:
            values_benchmark.append(agg_df_player_benchmark.iloc[0][vec_cols_agg_df_player[j]])


In [6]:
df_pizza = pd.DataFrame({
    'y_value' : values_to_chart,
    'y_bench' : values_benchmark,
    'Player_Attribute' : vec_cols_agg_df_player,
    'color_cat' : colors_bar,
})


In [7]:

fig = make_subplots(rows=1, cols=2)


fig1 = px.bar_polar(df_pizza, r='y_value', theta='Player_Attribute', color= 'Player_Attribute',
                   color_discrete_map={'finishing': "#9DEC1C",
                                       'crossing': "#1D9B28",
                                       'ball_control': "#49C718",
                                       'agility': "#217AEE",
                                       'stamina': "#6A68DF",
                                       'strength': "#52DAF1"
                                       })

fig2 = px.bar_polar(df_pizza, r='y_bench', theta='Player_Attribute', color= 'Player_Attribute',
                   color_discrete_map={'finishing': "#9DEC1C",
                                       'crossing': "#1D9B28",
                                       'ball_control': "#49C718",
                                       'agility': "#217AEE",
                                       'stamina': "#6A68DF",
                                       'strength': "#52DAF1"
                                       })

fig1.show()
fig2.show()

In [9]:
## Visualization as Dash App

app_1 = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
app_1.layout = dbc.Container([
    html.H1("CE_TM_05 - Player Physical and Sportive Condition", className="my-3"),
    html.H4("Select a Player to Analyze", className="my-3"),
    dbc.Row([
        dbc.Row([
            dcc.Dropdown(id='category-dropdown', options=df_player_att['player_api_id'].unique(), clearable=False),
            dbc.Button("Update Graph", id="update-btn", style={"backgroundColor": "#ff5733", "borderColor": "#ff5733"}, className="mt-2")
        ]),
        dbc.Row([
            dbc.Col([
                html.H4("Information about Selected Player", className="my-3"),
                dcc.Graph(id='graph_1'),
            ]),
            dbc.Col([
                html.H4("Rest of Plavers Average", className="my-3"),
                dcc.Graph(id='graph_2'),
            ])
        ])
    ])
])

@app_1.callback(
    [Output('graph_1', 'figure'),
    Output('graph_2', 'figure')],
    Input('update-btn', 'n_clicks'),
    State('category-dropdown', 'value'),
    prevent_initial_call=False
)

def update_graph(n_clicks, selected_player):

    df_player_example = df_player_att[df_player_att['player_api_id'] == selected_player]
    agg_df_player_example = df_player_example.groupby('player_api_id').agg(
        {'finishing' : 'mean',
        'crossing'  : 'mean',
        'ball_control'  : 'mean',
        'agility'  : 'mean',
        'stamina'  : 'mean',
        'strength'  : 'mean'}
    ).reset_index()

    df_player_benchmark = df_player_att[df_player_att['player_api_id'] != selected_player]
    agg_df_player_benchmark = df_player_benchmark.groupby('player_api_id').agg(
        {'finishing' : 'mean',
        'crossing'  : 'mean',
        'ball_control'  : 'mean',
        'agility'  : 'mean',
        'stamina'  : 'mean',
        'strength'  : 'mean'}
    ).reset_index()

    colors_bar = ["#1D9B28", "#49C718", "#9DEC1C", "#2439F3", "#0E56F1A9", "#5AD5EB" ]
    vars_play = ["finishing", "crossing", "ball_control", "agility", "stamina", "strength"]

    values_to_chart = []
    vec_cols_agg_df_player = agg_df_player_example.columns[1:].to_list()
    values_benchmark = []

    for i in range(0, len(vars_play)):
        for j in range(0, len(vec_cols_agg_df_player)):
            if vars_play[i] == vec_cols_agg_df_player[j]:
                values_to_chart.append(agg_df_player_example.iloc[0][vec_cols_agg_df_player[j]])

    for i in range(0, len(vars_play)):
        for j in range(0, len(vec_cols_agg_df_player)):
            if vars_play[i] == vec_cols_agg_df_player[j]:
                values_benchmark.append(agg_df_player_benchmark.iloc[0][vec_cols_agg_df_player[j]])

    df_pizza = pd.DataFrame({
        'y_value' : values_to_chart,
        'y_bench' : values_benchmark,
        'Player_Attribute' : vec_cols_agg_df_player,
        'color_cat' : colors_bar,
    })

    fig1 = px.bar_polar(df_pizza, r='y_value', theta='Player_Attribute', color= 'Player_Attribute',
                   color_discrete_map={'finishing': "#9DEC1C",
                                       'crossing': "#1D9B28",
                                       'ball_control': "#49C718",
                                       'agility': "#217AEE",
                                       'stamina': "#6A68DF",
                                       'strength': "#52DAF1"})

    fig2 = px.bar_polar(df_pizza, r='y_bench', theta='Player_Attribute', color= 'Player_Attribute',
                    color_discrete_map={'finishing': "#9DEC1C",
                                        'crossing': "#1D9B28",
                                        'ball_control': "#49C718",
                                        'agility': "#217AEE",
                                        'stamina': "#6A68DF",
                                        'strength': "#52DAF1"
                                        })

    return fig1, fig2

if __name__ == '__main__':
    app_1.run(port=8020)

[2026-05-18 05:47:06,758] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "c:\Users\pci\AppData\Local\Programs\Python\Python314\Lib\site-packages\flask\app.py", line 1511, in wsgi_app
    response = self.full_dispatch_request()
  File "c:\Users\pci\AppData\Local\Programs\Python\Python314\Lib\site-packages\flask\app.py", line 919, in full_dispatch_request
    rv = self.handle_user_exception(e)
  File "c:\Users\pci\AppData\Local\Programs\Python\Python314\Lib\site-packages\flask\app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
  File "c:\Users\pci\AppData\Local\Programs\Python\Python314\Lib\site-packages\flask\app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "c:\Users\pci\AppData\Local\Programs\Python\Python314\Lib\site-packages\da

In [ ]:
# 2. Import File

path_to_file = "https://raw.githubusercontent.com/cesarlarasantana/VDS_2526_G04_Football/refs/heads/main/Datasets"

df_country = pd.read_csv(path_to_file+"/Country.csv")
df_match_goals = pd.read_csv(path_to_file+"/Match_Goals.csv")
df_match_shots_on = pd.read_csv(path_to_file+"/Match_Shots_On.csv")
df_match_fouls = pd.read_csv(path_to_file+"/Match_Fouls_Committed.csv")
df_match_cards = pd.read_csv(path_to_file+"/Match_Cards.csv")
df_team = pd.read_csv(path_to_file+"/Team.csv")
df_match = pd.read_csv(path_to_file+"/Match.csv")
df_player = pd.read_csv(path_to_file+"/Player.csv")
df_player_att = pd.read_csv(path_to_file+"/Player_Attributes.csv", sep= ';')
df_position_ref = pd.read_csv(path_to_file+"/PositionReference.csv")

C:\Users\pci\AppData\Local\Temp\ipykernel_23524\3388209417.py:8: DtypeWarning: Columns (0: player1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_match_fouls = pd.read_csv(path_to_file+"/Match_Fouls_Committed.csv")
